In [1]:
import os
import re
import glob
import pandas as pd

# ------------------------------------------------------------------
# Input / output
# ------------------------------------------------------------------
INPUT_DIR = r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Plotting\CheckM_sum"
OUTPUT_COMPLETENESS = os.path.join(INPUT_DIR, "checkm_completeness_matrix.csv")
OUTPUT_GENOME_SIZE = os.path.join(INPUT_DIR, "checkm_genome_size_matrix.csv")

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def normalize_colname(name: str) -> str:
    """Normalize column names for matching."""
    name = str(name).replace("\n", " ").replace("\r", " ")
    name = re.sub(r"\s+", " ", name).strip().lower()
    return name

def build_col_lookup(columns):
    return {normalize_colname(c): c for c in columns}

def infer_method_name(filepath: str) -> str:
    """
    Infer assembly-method label from filename.
    Example:
      illumina_quality_report.tsv -> Illumina
      nanopore_quality_report.tsv -> Nanopore
      nanopore_racon_iteration3_quality_report.tsv -> Nanopore_racon
      nanopore_medaka_quality_report.tsv -> Nanopore_medaka
      nanopore_homopolisher_quality_report.tsv -> Nanopore_homopolisher
      nanopore_racon_medaka_quality_report.tsv -> Nanopore_racon_medaka
      nanopore_medaka_homopolisher_quality_report.tsv -> Nanopore_medaka_homopolisher
      hybrid_quality_report.tsv -> Hybrid
    """
    fname = os.path.basename(filepath).lower()

    mapping = {
        "illumina": "Illumina",
        "nanopore_quality_report": "Nanopore",
        "nanopore_racon_iteration3": "Nanopore_racon",
        "nanopore_medaka_quality_report": "Nanopore_medaka",
        "nanopore_homopolisher": "Nanopore_homopolisher",
        "nanopore_racon_medaka": "Nanopore_racon_medaka",
        "nanopore_medaka_homopolisher": "Nanopore_medaka_homopolisher",
        "hybrid": "Hybrid",
    }

    # ordered from more specific to less specific
    if "nanopore_medaka_homopolisher" in fname:
        return "Nanopore_medaka_homopolisher"
    elif "nanopore_racon_medaka" in fname:
        return "Nanopore_racon_medaka"
    elif "nanopore_homopolisher" in fname:
        return "Nanopore_homopolisher"
    elif "nanopore_medaka" in fname:
        return "Nanopore_medaka"
    elif "nanopore_racon_iteration3" in fname:
        return "Nanopore_racon"
    elif "nanopore_quality_report" in fname:
        return "Nanopore"
    elif "illumina" in fname:
        return "Illumina"
    elif "hybrid" in fname:
        return "Hybrid"

    # fallback: remove extension and common suffixes
    stem = os.path.splitext(os.path.basename(filepath))[0]
    stem = re.sub(r"_quality_report$", "", stem, flags=re.IGNORECASE)
    return stem

def read_report(filepath: str) -> pd.DataFrame:
    """
    Read one CheckM report, extracting Name, Completeness, Genome_size.
    Tries tab first, then comma.
    """
    # Try tab-delimited first
    try:
        df = pd.read_csv(filepath, sep="\t")
        if df.shape[1] == 1:
            df = pd.read_csv(filepath)
    except Exception:
        df = pd.read_csv(filepath)

    lookup = build_col_lookup(df.columns)

    required = ["name", "completeness", "genome_size"]
    missing = [c for c in required if c not in lookup]
    if missing:
        raise ValueError(
            f"Missing required columns {missing} in {filepath}\n"
            f"Available columns: {list(df.columns)}"
        )

    sub = df[
        [
            lookup["name"],
            lookup["completeness"],
            lookup["genome_size"],
        ]
    ].copy()

    sub.columns = ["Name", "Completeness", "Genome_size"]
    sub["Name"] = sub["Name"].astype(str).str.strip()
    sub = sub.drop_duplicates(subset=["Name"])

    return sub

def merge_one_field(filepaths, field_name):
    """
    Merge one field across all input files by Name.
    """
    merged = None
    used_names = set()

    for filepath in filepaths:
        method_name = infer_method_name(filepath)

        # avoid accidental duplicate column names
        original_method_name = method_name
        counter = 2
        while method_name in used_names:
            method_name = f"{original_method_name}_{counter}"
            counter += 1
        used_names.add(method_name)

        try:
            sub = read_report(filepath)[["Name", field_name]].copy()
        except Exception as e:
            print(f"Warning: could not process {filepath}: {e}")
            continue

        sub.columns = ["Name", method_name]

        if merged is None:
            merged = sub
        else:
            merged = pd.merge(merged, sub, on="Name", how="outer")

    if merged is None:
        return pd.DataFrame(columns=["Name"])

    merged = merged.sort_values("Name").reset_index(drop=True)
    return merged

# ------------------------------------------------------------------
# Find report files
# ------------------------------------------------------------------
patterns = ["*.tsv", "*.csv", "*.txt"]
report_files = []
for pattern in patterns:
    report_files.extend(glob.glob(os.path.join(INPUT_DIR, pattern)))

if not report_files:
    raise FileNotFoundError(f"No report files found in: {INPUT_DIR}")

print("Files detected:")
for f in report_files:
    print(" -", os.path.basename(f), "->", infer_method_name(f))

# ------------------------------------------------------------------
# Build outputs
# ------------------------------------------------------------------
completeness_df = merge_one_field(report_files, "Completeness")
genome_size_df = merge_one_field(report_files, "Genome_size")

# ------------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------------
completeness_df.to_csv(OUTPUT_COMPLETENESS, index=False)
genome_size_df.to_csv(OUTPUT_GENOME_SIZE, index=False)

print("\nDone.")
print("Completeness output:", OUTPUT_COMPLETENESS)
print("Genome size output:", OUTPUT_GENOME_SIZE)

Files detected:
 - hybrid_quality_report.tsv -> Hybrid
 - illumina_after_kraken_quality_report.tsv -> Illumina
 - illumina_quality_report.tsv -> Illumina
 - nanopore_after_kraken_quality_report.tsv -> nanopore_after_kraken
 - nanopore_homopolisher_quality_report.tsv -> Nanopore_homopolisher
 - nanopore_medaka_homopolisher_quality_report.tsv -> Nanopore_medaka_homopolisher
 - nanopore_medaka_quality_report.tsv -> Nanopore_medaka
 - nanopore_quality_report.tsv -> Nanopore
 - nanopore_racon_iteration3_quality_report.tsv -> Nanopore_racon
 - nanopore_racon_medaka_quality_report.tsv -> Nanopore_racon_medaka

Done.
Completeness output: C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Plotting\CheckM_sum\checkm_completeness_matrix.csv
Genome size output: C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Plotting\CheckM_sum\checkm_genome_size_matrix.csv
